A notebook that discusses relations constructed from manual annotation.

## Data and Libraries

In [1]:
import json

## JSON hierarchy loading

In [4]:
def print_relations_with_id(node):
    """
    Recursively traverses the hierarchy and prints relations that have a non-null ID.

    Args:
        node (dict): The current node in the hierarchy.
    """
    # Check if the current node has a non-null 'id'
    if node.get("id") is not None:
        # Print the ID and name, with padding for the ID for nice alignment
        print(f"ID: {node['id']:<4} | Name: {node['name']}")

    # If the node has children, recurse into each child
    if "children" in node and node["children"]:
        for child in node["children"]:
            print_relations_with_id(child)

In [5]:

# Load the JSON string into a Python dictionary
with open("rel_h.json", "r") as f:
    json_hierarchy = f.read()
data = json.loads(json_hierarchy)

print("--- Relations with a valid ID ---")

# Start the recursive traversal from the top-level node
print_relations_with_id(data)

--- Relations with a valid ID ---
ID: 2    | Name: is a
ID: 40   | Name: has type
ID: 84   | Name: has part
ID: 63   | Name: includes
ID: 89   | Name: consists of
ID: 10   | Name: chemical causes
ID: 25   | Name: causes
ID: 27   | Name: can cause
ID: 3    | Name: exposure can lead to
ID: 77   | Name: produces
ID: 85   | Name: causes variability to
ID: 111  | Name: causes anomalies in
ID: 53   | Name: causes a decrease in concentration of
ID: 68   | Name: causes a change in concentration of
ID: 168  | Name: causes an increase in concentration of
ID: 29   | Name: has effect on
ID: 76   | Name: has a positive effect on
ID: 4    | Name: has a negative health effect on
ID: 1    | Name: is a threat
ID: 62   | Name: has an unknown effect on
ID: 78   | Name: changes impact
ID: 79   | Name: possibly has effect on
ID: 139  | Name: has an indirect effect on
ID: 28   | Name: depends on
ID: 60   | Name: does not depend on
ID: 16   | Name: decreases
ID: 130  | Name: increases
ID: 138  | Name: intens

## Validation

In [6]:
import json
import re
from pathlib import Path

def parse_simple_data(filepath: Path) -> dict:
    """
    Parses the new, simplified RELdata_simple.txt file.

    Args:
        filepath: The path to the RELdata_simple.txt file.

    Returns:
        A dictionary mapping integer IDs to relation names.
    """
    if not filepath.exists():
        raise FileNotFoundError(f"Source of truth file not found at: {filepath}")

    relations = {}
    # Regex to find lines starting with '(ID) relation_name'
    # It captures the numeric ID and the rest of the line as the name.
    relation_pattern = re.compile(r'^\s*\((\d+)\)\s*(.*)')

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            # Skip empty lines or header
            if not line.strip() or line.strip().lower() == 'relation':
                continue

            match = relation_pattern.match(line)
            if match:
                rel_id = int(match.group(1))
                # .strip() removes leading/trailing whitespace from the name
                rel_name = match.group(2).strip()
                
                # We only validate relations that have a name in the source file.
                if rel_name:
                    relations[rel_id] = rel_name
    return relations

def parse_json_hierarchy(filepath: Path) -> dict:
    """
    Loads the JSON hierarchy and extracts all relations with a non-null ID.

    Args:
        filepath: The path to the hierarchy.json file.

    Returns:
        A dictionary mapping integer IDs to relation names.
    """
    if not filepath.exists():
        raise FileNotFoundError(f"JSON hierarchy file not found at: {filepath}")

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    relations = {}
    def traverse(node):
        """Recursively walk the hierarchy."""
        if node.get("id") is not None:
            relations[node["id"]] = node["name"].strip()
        if "children" in node:
            for child in node["children"]:
                traverse(child)
    traverse(data)
    return relations

def validate_relations(source_relations: dict, json_relations: dict):
    """
    Compares the two dictionaries of relations and prints a validation report.
    """
    source_ids = set(source_relations.keys())
    json_ids = set(json_relations.keys())

    missing_in_json = source_ids - json_ids
    extra_in_json = json_ids - source_ids
    common_ids = source_ids.intersection(json_ids)

    name_discrepancies = []
    for rel_id in common_ids:
        source_name = source_relations[rel_id]
        json_name = json_relations[rel_id]
        if source_name != json_name:
            name_discrepancies.append({
                "id": rel_id,
                "source": source_name,
                "json": json_name
            })

    # --- Print Report ---
    print("--- Relation Hierarchy Validation Report ---")
    print(f"\nFound {len(source_relations)} named relations in '{source_data_path.name}'")
    print(f"Found {len(json_relations)} relations with IDs in '{json_data_path.name}'")
    print("-" * 40)

    if not missing_in_json and not extra_in_json and not name_discrepancies:
        print("\n✅ SUCCESS: The JSON hierarchy perfectly matches the source of truth file.")
        return

    if missing_in_json:
        print(f"\n❌ WARNING: {len(missing_in_json)} relations are in '{source_data_path.name}' but MISSING from JSON.")
        print("   (This is expected if the relation was undefined in the original detailed file).")
        for rel_id in sorted(list(missing_in_json)):
            print(f"  - ID: {rel_id:<4} | Name: '{source_relations[rel_id]}'")
    
    if extra_in_json:
        print(f"\n❌ ERROR: {len(extra_in_json)} relations are in the JSON but NOT in '{source_data_path.name}'.")
        for rel_id in sorted(list(extra_in_json)):
            print(f"  - ID: {rel_id:<4} | Name: '{json_relations[rel_id]}'")
    
    if name_discrepancies:
        print(f"\n❌ ERROR: {len(name_discrepancies)} relations have name mismatches.")
        for item in name_discrepancies:
            print(f"  - ID: {item['id']}")
            print(f"    Source File: '{item['source']}'")
            print(f"    JSON File:   '{item['json']}'")

if __name__ == "__main__":
    try:
        source_data_path = Path("/home/p0l3/RAD/DROP/RELdata_simple.txt")
        json_data_path = Path("rel_h.json")

        source_rels = parse_simple_data(source_data_path)
        json_rels = parse_json_hierarchy(json_data_path)

        validate_relations(source_rels, json_rels)

    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure 'RELdata_simple.txt' and 'hierarchy.json' are in the same directory.")

--- Relation Hierarchy Validation Report ---

Found 198 named relations in 'RELdata_simple.txt'
Found 93 relations with IDs in 'rel_h.json'
----------------------------------------

❌ WARNING: 106 relations are in 'RELdata_simple.txt' but MISSING from JSON.
   (This is expected if the relation was undefined in the original detailed file).
  - ID: 6    | Name: 'used for (time span)'
  - ID: 17   | Name: 'has attribute (target range)'
  - ID: 18   | Name: 'has attribute (recommended dose)'
  - ID: 19   | Name: 'has a positive health effect on'
  - ID: 20   | Name: 'has attribute (intake interval)'
  - ID: 23   | Name: 'has habitat'
  - ID: 24   | Name: 'has an increase in'
  - ID: 33   | Name: 'obtained in (time)'
  - ID: 37   | Name: 'part of'
  - ID: 38   | Name: 'appliance method'
  - ID: 42   | Name: 'performed measurement of'
  - ID: 43   | Name: 'has elevation of clinical significance at'
  - ID: 54   | Name: 'has attribute (reference range)'
  - ID: 56   | Name: 'has a therapeutic